### 피드 포워드 신경망 언어 모델(Neural Network Language Model, NNLM)


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim


# ==========================================
# 1. NNLM (Neural Network Language Model) 클래스 정의
# ==========================================
class NNLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, block_size, hidden_dim):
        """
        NNLM 레이어 초기화 및 파라미터(Weight) Shape 정의:

        - vocab_size: 17
        - embedding_dim: 8
        - block_size: 3
        - hidden_dim: 16
        """
        super(NNLM, self).__init__()

        # 1. 단어 임베딩 룩업 테이블
        # self.C.weight shape: [vocab_size, embedding_dim] -> [17, 8]
        self.C = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

        # 2. 은닉층 선형 변환 레이어 (입력 차원: block_size * embedding_dim = 3 * 8 = 24)
        # self.hidden.weight shape: [hidden_dim, block_size * embedding_dim] -> [16, 24]
        # self.hidden.bias shape:   [hidden_dim] -> [16]
        self.hidden = nn.Linear(
            in_features=block_size * embedding_dim, out_features=hidden_dim
        )

        # 비선형 활성화 함수 (차원 변화 없음)
        self.tanh = nn.Tanh()

        # 3. 은닉층 -> 출력층 선형 변환 레이어
        # self.output.weight shape: [vocab_size, hidden_dim] -> [17, 16]
        # self.output.bias shape:   [vocab_size] -> [17]
        self.output = nn.Linear(in_features=hidden_dim, out_features=vocab_size)

        # 4. Direct Connection (임베딩 결합층 -> 출력층 직결 레이어, bias=False)
        # self.direct.weight shape: [vocab_size, block_size * embedding_dim] -> [17, 24]
        self.direct = nn.Linear(
            in_features=block_size * embedding_dim, out_features=vocab_size, bias=False
        )

    def forward(self, x):
        """
        [학습 시 Tensor Shape 변화] (Batch Size B = 15 가정)
        Input x shape: [B, block_size] -> [15, 3]
        """
        # embeds shape: [B, block_size, embedding_dim] -> [15, 3, 8]
        embeds = self.C(x)

        # concat_embeds shape: [B, block_size * embedding_dim] -> [15, 24]
        concat_embeds = embeds.view(embeds.size(0), -1)

        # h shape: [B, hidden_dim] -> [15, 16]
        h = self.tanh(self.hidden(concat_embeds))

        # self.output(h) shape:         [B, vocab_size] -> [15, 17]
        # self.direct(concat_embeds) shape: [B, vocab_size] -> [15, 17]
        # logits shape:                 [B, vocab_size] -> [15, 17]
        logits = self.output(h) + self.direct(concat_embeds)

        return logits


# ==========================================
# 2. 텍스트 데이터 전처리 함수
# ==========================================
def make_dataset(text, word2idx, block_size):
    words = text.split()  # len(words) = 18
    X, Y = [], []

    # 총 (18 - 3) = 15개의 슬라이딩 윈도우 데이터 생성
    for i in range(len(words) - block_size):
        context = [word2idx[w] for w in words[i : i + block_size]]
        target = word2idx[words[i + block_size]]
        X.append(context)
        Y.append(target)

    # Return X shape: [15, 3]
    # Return Y shape: [15]
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)


# ==========================================
# 3. 모델 학습 및 파이프라인
# ==========================================
if __name__ == "__main__":
    torch.manual_seed(42)

    sample_text = (
        "the neural network language model predicts the next word "
        "given a sequence of previous words using embedding vectors"
    )

    # tokens length: 17 (중복 제거 후 고유 단어 수)
    tokens = sorted(list(set(sample_text.split())))

    # vocab_size = 17
    vocab_size = len(tokens)

    word2idx = {w: i for i, w in enumerate(tokens)}
    idx2word = {i: w for i, w in enumerate(tokens)}

    # 하이퍼파라미터 설정
    BLOCK_SIZE = 3  # Context Window
    EMBEDDING_DIM = 8  # 임베딩 차원
    HIDDEN_DIM = 16  # 은닉층 노드 수
    LEARNING_RATE = 0.01
    EPOCHS = 200

    # 데이터셋 생성
    # X shape: [15, 3]  (Batch Size = 15, Block Size = 3)
    # Y shape: [15]     (Batch Size = 15)
    X, Y = make_dataset(sample_text, word2idx, BLOCK_SIZE)

    # NNLM 모델 객체 생성
    model = NNLM(vocab_size, EMBEDDING_DIM, BLOCK_SIZE, HIDDEN_DIM)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print("=== NNLM 학습 시작 ===")
    for epoch in range(1, EPOCHS + 1):
        optimizer.zero_grad()

        # logits shape: [15, 17]
        logits = model(X)

        # loss: 스칼라 (Scalar, shape: [])
        loss = criterion(logits, Y)

        loss.backward()
        optimizer.step()

        if epoch % 40 == 0:
            print(f"Epoch [{epoch:3d}/{EPOCHS}] - Loss: {loss.item():.4f}")

    # ==========================================
    # 4. 추론 (Inference) 테스트
    # ==========================================
    print("\n=== 추론 (다음 단어 예측) 테스트 ===")
    model.eval()
    with torch.no_grad():
        test_context_str = ["sequence", "of", "previous"]

        # test_context_idx shape: [1, 3] (Batch Size = 1, Block Size = 3)
        test_context_idx = torch.tensor(
            [[word2idx[w] for w in test_context_str]], dtype=torch.long
        )

        # output_logits shape: [1, 17]
        output_logits = model(test_context_idx)

        # predicted_idx: 스칼라 정수값 (e.g., 16)
        predicted_idx = torch.argmax(output_logits, dim=-1).item()
        predicted_word = idx2word[predicted_idx]

        print(f"입력 문맥 (Context Window) : {' '.join(test_context_str)}")
        print(f"모델 예측 결과 (Next Word) : {predicted_word}")

=== NNLM 학습 시작 ===
Epoch [ 40/200] - Loss: 0.0057
Epoch [ 80/200] - Loss: 0.0025
Epoch [120/200] - Loss: 0.0018
Epoch [160/200] - Loss: 0.0014
Epoch [200/200] - Loss: 0.0011

=== 추론 (다음 단어 예측) 테스트 ===
입력 문맥 (Context Window) : sequence of previous
모델 예측 결과 (Next Word) : words
